# Additional API demonstrations

Screen three abstracts, extract the sample article and supporting information, and reconstruct negative-condition plans. Install `.[mining,notebook]` from the repository root before opening this notebook.

The live switches default to `False`. When a switch is enabled, provide your API key at the hidden prompt, or set `OPENAI_API_KEY` in the environment. Requests use the canonical prompts and the model settings in this folder's `configs/`. Outputs are saved under `results/examples/additional_demo_api_needed/`.

## Inputs and setup

Run the cells in order. Paths in this table are relative to `Demo/additional_demo_api_needed/`, except the `Demo/05_data_mining/` paths, which start at the repository root.

| Input | Location | Supplied content and replacement |
| --- | --- | --- |
| Abstract metadata, CSV | `inputs/triage_metadata.csv` | Three bibliographic records. For other papers, retain the `DOI`, title, source, keyword, and `Abstract` columns. |
| Triage reference, CSV | `inputs/triage_ground_truth.csv` | Corresponding `DOI` and `Consensus GT` labels, with annotation columns. Replace together with the metadata for another reference set. |
| Illustrative article/SI pair, PDF | `Demo/05_data_mining/articles/`, `Demo/05_data_mining/supporting_information/` | Ready for a small extraction demonstration with the default mining configuration. |
| Real-paper DOI template, CSV | `literature_input/inventory.csv` | Three DOI rows. Edit this list to select another three to five papers. |
| Main-article placeholders, PDF | `literature_input/main/` | Replace each with the real article PDF matching its DOI filename. |
| SI placeholders, PDF | `literature_input/si/` | Replace each with the real SI PDF; keep the DOI filename and `_SI` suffix. |

`MINING_CONFIG_DIR` below selects the document set. Keep `DEMO / "configs"` for the included illustrative pair. Select `DEMO / "configs/local_papers"` to use your replacement PDFs. Placeholder PDFs are only filename templates; validation lists them and live extraction requires their replacement. For another DOI, name the article `10.xxxx_example.pdf` and its SI `10.xxxx_example_SI.pdf`, and update `literature_input/inventory.csv` accordingly.

Triage uses `configs/triage.json` in either mode. Enable each live switch only for the stage being run. Positive mining creates the CSV and complete synthesis JSON store required by negative mining. Run positive mining before enabling negative mining. The real-paper configuration saves its manifest and extraction outputs under `results/examples/additional_demo_api_needed/local_papers/`.

Implementation: [demo runner](run_demo.py), [positive extraction](../../src/mofinder/extraction/positive.py), and [negative extraction](../../src/mofinder/extraction/negative.py). See the [source-to-code guide](../../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/mofinder").is_dir()
)
DEMO = ROOT / "Demo/additional_demo_api_needed"
sys.path.insert(0, str(DEMO))
from run_demo import validate_all, run_triage, run_positive, run_negative

MINING_CONFIG_DIR = DEMO / "configs"
# For replacement article/SI PDFs: MINING_CONFIG_DIR = DEMO / "configs/local_papers"

RUN_TRIAGE = False
RUN_POSITIVE_MINING = False
RUN_NEGATIVE_MINING = False

print(json.dumps(validate_all(MINING_CONFIG_DIR), indent=2))


## 1. Literature triage

The input files contain the first two Y papers and first N paper from the repository's twelve-paper triage example, using one model and one round. The human labels remain reference data and are not included in the model prompt. This small example demonstrates execution; it is not a performance benchmark.


In [ ]:
if RUN_TRIAGE:
    triage_run = await run_triage()
    print("Triage results:", triage_run.output_dir)


## 2. Positive data mining

The default article/SI pair in `Demo/05_data_mining/` contains illustrative syntheses. The `local_papers` configuration reads the replacement documents in `literature_input/`. Positive extraction saves a CSV and the per-synthesis JSON files used by the next stage.


In [ ]:
if RUN_POSITIVE_MINING:
    print(json.dumps(run_positive(MINING_CONFIG_DIR), indent=2))


## 3. Negative data mining

Run positive extraction first. Negative mining selects documents marked `YES` for trial or failure evidence, creates modification plans, and enumerates their saved options locally. The extracted evidence determines whether negative records are produced. Enumerated combinations are reconstructed conditions, not independent measurements of experimental failure.


In [ ]:
if RUN_NEGATIVE_MINING:
    print(json.dumps(run_negative(live=True, config_dir=MINING_CONFIG_DIR), indent=2, default=str))
